# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. 

### Dataset Source
The dataset schema is provided via a Croissant JSON-LD URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant` and display key information.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print basic dataset information
print(f"Title: {dataset.metadata.name}")
print(f"Version: {dataset.metadata.version}")
print(f"Description: {dataset.metadata.description}")
print(f"License: {dataset.metadata.license}")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Review available record sets and associated fields. For all entity references, use their `@id` as required for reproducible exploration.

Let's list the record sets (`cr:RecordSet`), then for each record set, list its fields (`cr:field`) and columns (`cr:column`) and their `@id`s.

In [ ]:
# Extract all record sets by their @id
record_sets = dataset.metadata.recordSet or []

if not record_sets:
    print("No record sets found in the schema.")
else:
    print(f"Found {len(record_sets)} record set(s):\n")
    for idx, rs in enumerate(record_sets):
        print(f"[{idx}] RecordSet: {rs['@id']}")
        print(f"  Name: {rs.get('name', 'N/A')}")
        # List fields (features, columns)
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields (by @id):")
        for field in fields:
            if isinstance(field, dict):
                print(f"    - {field.get('@id', field)} (label: {field.get('name', 'N/A')})")
            else:
                print(f"    - {field}")
        columns = rs.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        if columns:
            print("  Columns (by @id):")
            for col in columns:
                if isinstance(col, dict):
                    print(f"    - {col.get('@id', col)} (label: {col.get('name', 'N/A')})")
                else:
                    print(f"    - {col}")
        print()

## 3. Data Extraction
Load data from each record set into a pandas `DataFrame`.

Specify the record set and field `@id`s from the above step to extract and inspect data.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets] if record_sets else []
dataframes = {}

for rs_id in record_set_ids:
    # Load records for this record set
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for record set {rs_id}")
    else:
        print(f"No records found for record set {rs_id}")

# For demonstration: use the first record set with data
main_rs_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        main_rs_id = rs_id
        break
if main_rs_id:
    print(f"\nFields/columns for record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    print("\nSample records:")
    display(dataframes[main_rs_id].head())
else:
    print("No data found in any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data analysis steps using the primary record set and its numeric fields. 

**Note:** Replace `<numeric_field_id>` with a numeric field or column `@id` as determined from the columns listed above. 

In [ ]:
import numpy as np

if main_rs_id:
    df = dataframes[main_rs_id]
    # Guess a numeric field
    candidate_numeric_fields = [col for col in df.columns if df[col].dtype in [np.float32, np.float64, np.int32, np.int64, float, int, 'int64', 'float64', 'int32'] or pd.api.types.is_numeric_dtype(df[col])]
    if candidate_numeric_fields:
        numeric_field = candidate_numeric_fields[0]
        print(f"Automatically selected numeric_field: {numeric_field}")
        # Choose a threshold: use median as illustration
        threshold = df[numeric_field].median() if np.issubdtype(df[numeric_field].dtype, np.number) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (N={len(filtered_df)}):")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouped aggregation
        # Try to find a group field (e.g. a string/categorical field)
        candidate_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
        if candidate_group_fields:
            group_field = candidate_group_fields[0]
            print(f"\nGrouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field detected for EDA.")
else:
    print("No extracted data available for analysis.")

## 5. Visualization
Visualize numeric data distributions and groupwise statistics from the main record set using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and candidate_numeric_fields:
    # Plot histogram of the numeric field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in {main_rs_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    # If grouping was done
    if 'grouped_df' in locals():
        plt.figure(figsize=(10,5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Cannot plot: no numeric field or data available.")

## 6. Conclusion
We demonstrated how to load a dataset described by a Croissant schema via its URL using `mlcroissant`.

- The dataset's record sets and fields were introspected using only their `@id` identifiers per FAIR principles and interoperability.
- Initial exploratory data analysis was performed, and simple visualizations generated for the main numeric field present in the data.
- This workflow can be customized for further, model-specific analysis by referencing all fields using their `@id`.

For more advanced usage, refer to [`mlcroissant` documentation](https://github.com/mlcommons/croissant).